This notebook is set up to test code as needed.

In [ ]:
# import packages
import pandas as pd
import os

%reload_ext autoreload
%autoreload 2

# # Tell python where to look for modules.
import sys
sys.path.append('../../../open-grid-emissions/src/')

import download_data
import load_data
from column_checks import get_dtypes
from filepaths import *
import impute_hourly_profiles
import data_cleaning
import output_data
import emissions
import validation
import gross_to_net_generation
import eia930
import consumed

year = 2020
path_prefix = "" 
path_prefix += f"{year}/"

## Investigate power sector data

In [ ]:
resolution = "hourly"

all_data = []
for ba in os.listdir(results_folder(f"2020/power_sector_data/{resolution}/us_units")):
    df = pd.read_csv(results_folder(f"2020/power_sector_data/{resolution}/us_units/{ba}"))
    df["ba_code"] = ba.split(".")[0]
    all_data.append(df)

all_data = pd.concat(all_data, axis=0)


In [ ]:
# identify how many negative values there are
for col in ['net_generation_mwh', 'fuel_consumed_mmbtu', 'fuel_consumed_for_electricity_mmbtu', 'co2_mass_lb', 'ch4_mass_lb', 'n2o_mass_lb', 'co2e_mass_lb', 'nox_mass_lb', 'so2_mass_lb', 'co2_mass_lb_for_electricity', 'ch4_mass_lb_for_electricity', 'n2o_mass_lb_for_electricity', 'co2e_mass_lb_for_electricity', 'nox_mass_lb_for_electricity', 'so2_mass_lb_for_electricity', 'co2_mass_lb_adjusted', 'ch4_mass_lb_adjusted', 'n2o_mass_lb_adjusted', 'co2e_mass_lb_adjusted', 'nox_mass_lb_adjusted', 'so2_mass_lb_adjusted', 'co2_mass_lb_for_electricity_adjusted', 'ch4_mass_lb_for_electricity_adjusted', 'n2o_mass_lb_for_electricity_adjusted', 'co2e_mass_lb_for_electricity_adjusted', 'nox_mass_lb_for_electricity_adjusted', 'so2_mass_lb_for_electricity_adjusted', 'generated_co2_rate_lb_per_mwh_for_electricity', 'generated_ch4_rate_lb_per_mwh_for_electricity', 'generated_n2o_rate_lb_per_mwh_for_electricity', 'generated_co2e_rate_lb_per_mwh_for_electricity',
       'generated_nox_rate_lb_per_mwh_for_electricity', 'generated_so2_rate_lb_per_mwh_for_electricity', 'generated_co2_rate_lb_per_mwh_for_electricity_adjusted', 'generated_ch4_rate_lb_per_mwh_for_electricity_adjusted', 'generated_n2o_rate_lb_per_mwh_for_electricity_adjusted', 'generated_co2e_rate_lb_per_mwh_for_electricity_adjusted', 'generated_nox_rate_lb_per_mwh_for_electricity_adjusted', 'generated_so2_rate_lb_per_mwh_for_electricity_adjusted']:
    if len(all_data[all_data[col] < 0]) > 0:
        print(f"There are {len(all_data[all_data[col] < 0])} negative {col} values")
        print(all_data.loc[all_data[col] < 0, "ba_code"].unique())


## Investigate carbon accounting data

In [ ]:
resolution = "hourly"

all_data = []
for ba in os.listdir(results_folder(f"2020/carbon_accounting/{resolution}/us_units")):
    df = pd.read_csv(results_folder(f"2020/carbon_accounting/{resolution}/us_units/{ba}"))
    df["ba_code"] = ba.split(".")[0]
    all_data.append(df)

all_data = pd.concat(all_data, axis=0)

In [ ]:
# identify how many negative values there are
for col in ['consumed_co2_rate_lb_per_mwh_for_electricity', 'consumed_ch4_rate_lb_per_mwh_for_electricity', 'consumed_n2o_rate_lb_per_mwh_for_electricity', 'consumed_co2e_rate_lb_per_mwh_for_electricity', 'consumed_nox_rate_lb_per_mwh_for_electricity', 'consumed_so2_rate_lb_per_mwh_for_electricity', 'consumed_co2_rate_lb_per_mwh_for_electricity_adjusted', 'consumed_ch4_rate_lb_per_mwh_for_electricity_adjusted', 'consumed_n2o_rate_lb_per_mwh_for_electricity_adjusted', 'consumed_co2e_rate_lb_per_mwh_for_electricity_adjusted', 'consumed_nox_rate_lb_per_mwh_for_electricity_adjusted', 'consumed_so2_rate_lb_per_mwh_for_electricity_adjusted']:
    if len(all_data[all_data[col] < 0]) > 0:
        print(f"There are {len(all_data[all_data[col] < 0])} negative {col} values")
        print(all_data.loc[all_data[col] < 0, "ba_code"].unique())


## Observations
For 2020, there is no negative power sector data at the monthly or annual resolution. There are negative rates in the hourly data
['AEC' 'IID' 'NEVP' 'NYIS' 'OHMS']

The carbon accounting data has negative values for:
- Annual: 'PGE', 'PSEI', 'SCL', 'TPWR'
- Monthly: ['PACW' 'PGE' 'PSEI' 'SCL' 'TPWR']
 - Hourly: ['CPLW']

After fixing net consumed mwh calc:
- Annual: None
- Monthly: ["EEI"]
 - Hourly: ["CPLW"]

## Investigate consumption calculation

In [ ]:
# create a test folder to hold these results
for unit in ["us_units", "metric_units"]:
    for time_resolution in output_data.TIME_RESOLUTIONS.keys():
        for subfolder in ["carbon_accounting_TEST"]:
            os.makedirs(
                results_folder(
                    f"{path_prefix}/{subfolder}/{time_resolution}/{unit}"
                ),
                exist_ok=True,
            )

In [ ]:

clean_930_file = outputs_folder(f"{path_prefix}/eia930/eia930_elec.csv")
hourly_consumed_calc = consumed.HourlyConsumed(
        clean_930_file,
        path_prefix,
        year,
        small=False,
        skip_outputs=False,
    )
hourly_consumed_calc.run()
hourly_consumed_calc.output_results()